# Arabic Cursive OCR Benchmark — Incremental Local Sweep

Runs the local models (Qari, Qwen-VL, TrOCR, …) across **every** benchmark
dataset, one batch of 10 new lines per (model, dataset) each time you run
it. **Run all, walk away, come back and Run all again** to add the next 10.
After every (model, dataset) batch it rebuilds the leaderboard, mirrors to
Drive, and pushes to the repo — a disconnect never loses more than the
model in flight.

**Incremental + resumable:** already-scored lines come from
`results/_cache/` and are never recomputed; a (model, dataset) whose split
is fully covered is skipped entirely. A model that OOMs, a dataset that
won't download, a line the model refuses — each is caught and logged; the
sweep keeps going. Safe to Run all again after a timeout.

**Before running:** `Runtime -> T4 GPU`; add `GH_TOKEN` (write-scoped) and
`HF_TOKEN` secrets (see `notebooks/README.md`).

## 1. GPU (falls back to CPU)

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

## 3. Install + caches

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[local]'
# Colab ships an old torchao; recent transformers hard-errors importing it
# at model-load ('incompatible version of torchao'). We don't use it.
!pip uninstall -y -q torchao 2>/dev/null; echo 'torchao removed (unused)'

## 4. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Config

**Incremental by default.** With `ACCUMULATE = True`, each run of this
notebook scores exactly one fresh batch of `BATCH` (= 10) lines for every
(model, dataset) pair that still has un-scored lines, retrying anything that
errored before. A model that has already covered a whole split is skipped —
no empty run, no empty commit. Nothing already scored is recomputed
(per-line cache in `results/_cache/`, pushed with the results). So the loop
is: **Run all → get 10 more lines per pair → Run all again → 10 more …**
until every split is fully covered.

`DATASETS` lists every benchmark set: the KITAB-Bench subsets
(`kitab_khatt`, `kitab_muharaf`, `kitab_historicalbooks`, `kitab_hindawi`,
`kitab_historyar`, `kitab_onlinekhatt`) plus `muharaf_lines` (1,334 lines,
needs `HF_TOKEN`). Drop any you don't want.

Set `ACCUMULATE = False` for the old two-phase behaviour: a `PARTIAL_LIMIT`
quick pass per model, then the whole split when `RUN_FULL = True`, each
phase checkpointed in the manifest and skipped once done.

The 7B models (`qwen2.5-vl-7b`, `ain-7b`) load in **4-bit** — fine on a
T4. **On CPU** (no GPU) 4-bit isn't supported and full fp32 7B OOMs the
12 GB box, so they're dropped and the batch shrinks — enough to prove the
pipeline, not a real benchmark.

In [ ]:
MODELS = [
    'qari-ocr-0.2',    # Arabic OCR, 2B, raw-text focused (lead)
    'qari-ocr-0.3',    # Arabic OCR, 2B, structure-first (emits markup)
    'qwen2-vl-2b',     # untuned base — anchor
    'qwen2.5-vl-3b',   # general VLM, 3B
    'qwen2.5-vl-7b',   # general VLM, 7B (4-bit on GPU)
    'ain-7b',          # Arabic-centric VLM, 7B (4-bit on GPU)
    'minicpm-v',       # multilingual doc-OCR VLM, 8B (4-bit on GPU)
    'trocr-large-arabic',  # dedicated HTR (TrOCR-Large, KHATT+IAM); CPU-ok
]
# Every benchmark dataset. Each re-run of this notebook adds ONE fresh batch
# of BATCH lines per (model, dataset) that has un-scored lines left; a model
# that has already covered a whole split is skipped (no empty run/commit).
DATASETS = [
    'kitab_khatt', 'kitab_muharaf', 'kitab_historicalbooks', 'kitab_hindawi',
    'kitab_historyar', 'kitab_onlinekhatt',    # newer KITAB-Bench subsets
    'muharaf_lines',                           # 1,334 lines, needs HF_TOKEN
]
BATCH = 10               # NEW lines scored per (model, dataset) per notebook run
ACCUMULATE = True        # incremental: add BATCH new lines each run, retry past errors,
                         # recompute nothing already scored. Set False for the old
                         # partial-then-full behaviour below.
PARTIAL_LIMIT = BATCH    # (non-accumulate) quick pass size per model
RUN_FULL = False         # (non-accumulate) then the whole split for each dataset

import yaml
_mc = yaml.safe_load(open('configs/models.yaml'))
if not HAS_GPU:
    # CPU: drop 4-bit models (GPU-only), and anything whose min_ram_gb
    # exceeds free RAM — loading that would OOM-kill the kernel. The
    # client also re-checks this at load time, so a model that slips
    # through is skipped cleanly (logged, sweep continues), not crashed.
    import psutil
    _free = psutil.virtual_memory().available / 1e9
    _keep = []
    for m in MODELS:
        c = _mc.get(m, {})
        if c.get('load_in_4bit'):
            print(f'  drop {m}: 4-bit, GPU-only'); continue
        need = float(c.get('min_ram_gb', 0) or 0)
        if need and need > _free:
            print(f'  drop {m}: needs ~{need:.0f} GB, only {_free:.1f} GB free'); continue
        _keep.append(m)
    MODELS = _keep
    PARTIAL_LIMIT = min(PARTIAL_LIMIT, 10)
    RUN_FULL = False
    print(f'CPU mode ({_free:.1f} GB free) -> MODELS={MODELS}, PARTIAL_LIMIT={PARTIAL_LIMIT}')

In [ ]:
# bitsandbytes only if a 4-bit model is in the list (GPU mode)
if HAS_GPU and any(_mc.get(m, {}).get('load_in_4bit') for m in MODELS):
    !pip install -q bitsandbytes
    print('bitsandbytes installed for 4-bit models')

## 6. Run the sweep

One batch of `BATCH` new lines per (model, dataset) with un-scored lines
left. Run this cell again (or Run all) to add the next batch.

In [ ]:
import os, sys, shutil, subprocess, pathlib
# make the src-layout package importable even if `pip install -e` didn't take
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from arabic_ocr_bench.sweep import run_sweep
from arabic_ocr_bench.samples import export_run_samples

DRIVE = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
DRIVE_OK = DRIVE.parent.exists()   # is Drive mounted?

_DS = {}
_HF_SPLIT = {
    'muharaf_lines': ('aamijar/muharaf-public', 'test'),
    'kitab_khatt': ('ahmedheakl/arocrbench_khatt', 'train'),
    'kitab_muharaf': ('ahmedheakl/arocrbench_muharaf', 'train'),
    'kitab_historicalbooks': ('ahmedheakl/arocrbench_historicalbooks', 'train'),
    'kitab_hindawi': ('ahmedheakl/arocrbench_hindawi', 'train'),
    'kitab_historyar': ('ahmedheakl/arocrbench_historyar', 'train'),
    'kitab_onlinekhatt': ('ahmedheakl/arocrbench_onlinekhatt', 'train'),
}  # add datasets here (must match configs/datasets.yaml)
def _get_image_fn(dataset):
    # row-index -> PIL image, for the sample export
    if dataset not in _HF_SPLIT:
        return lambda i: None   # unknown dataset -> samples show 'image unavailable'
    if dataset not in _DS:
        from datasets import load_dataset
        hf_id, split = _HF_SPLIT[dataset]
        _DS[dataset] = load_dataset(hf_id, split=split)
    ds = _DS[dataset]
    return lambda i: ds[int(i)]['image']

def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True)

def _mirror_to_drive(run_dir):
    if not DRIVE_OK:
        return
    (DRIVE / 'results').mkdir(parents=True, exist_ok=True)
    if pathlib.Path('results/_cache').is_dir():
        shutil.copytree('results/_cache', DRIVE / '_cache', dirs_exist_ok=True)
    shutil.copytree(run_dir, DRIVE / 'results' / pathlib.Path(run_dir).name, dirs_exist_ok=True)
    for f in ('sweep_manifest.json', 'leaderboard.html', 'leaderboard.json'):
        if pathlib.Path('results', f).is_file():
            shutil.copy2(pathlib.Path('results', f), DRIVE / 'results' / f)

def checkpoint(entry, run_dir):
    # Called by run_sweep AFTER a phase is scored. Persist the results here.
    # RAISE if they don't make it to the repo -> the phase is rolled back and
    # retried next run (so a 'done' entry always means 'in the repo').
    run_dir = str(run_dir)
    try:
        export_run_samples(run_dir, get_image=_get_image_fn(entry['dataset']), k=3)
    except Exception as e:
        print('   sample export failed:', e)
    subprocess.run(['python', 'scripts/make_leaderboard.py', 'results',
                    '-o', 'results/leaderboard.html'])
    _mirror_to_drive(run_dir)
    if not token:
        print('   no GH_TOKEN -> Drive only (this counts as persisted)')
        return
    _git('config', 'user.email', 'colab@arabic-ocr-bench')
    _git('config', 'user.name', 'colab-sweep')
    _git('add', '-f', run_dir, 'results/_cache', 'results/sweep_manifest.json',
         'results/leaderboard.html', 'results/leaderboard.json')
    _git('commit', '-m', f"sweep: {entry['model']} {entry['phase']} "
         f"CER-light {entry['cer_micro_light']*100:.1f}% n={entry['n']}")
    _git('pull', '--rebase', '--autostash', 'origin', 'main')
    p = _git('push', 'origin', 'HEAD:main')
    if p.returncode != 0:
        raise RuntimeError('git push failed: ' + (p.stderr.strip()[:300] or 'unknown'))
    print('   push: ok')

# The manifest is git-tracked (comes from the fresh checkout) — it is the
# single source of truth for what's done. Only restore the per-line cache
# from Drive, to make a re-running phase fast.
if DRIVE_OK and (DRIVE / '_cache').is_dir():
    shutil.copytree(DRIVE / '_cache', 'results/_cache', dirs_exist_ok=True)
    print('restored line cache from Drive')

# One run_sweep per dataset. Each is independently resumable and wrapped so
# a dataset that fails outright (e.g. a gated download) doesn't stop the rest.
# Per-model / per-phase / per-line failures are already caught inside run_sweep.
for _ds in DATASETS:
    print(f'\n########## DATASET: {_ds} ##########')
    try:
        run_sweep(
            models=MODELS, dataset=_ds,
            partial_limit=PARTIAL_LIMIT, run_full=RUN_FULL,
            accumulate=ACCUMULATE,
            concurrency=1,
            manifest_path='results/sweep_manifest.json', results_dir='results',
            on_progress=print, on_done=checkpoint,
        )
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'!!! DATASET {_ds} FAILED ({type(e).__name__}: {e}) — continuing !!!')
print('\n=== SWEEP COMPLETE ===')

## 7. Final leaderboard

In [ ]:
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## Notes

- **Add the next batch:** just Run all again. Each (model, dataset) gets 10
  more lines; scored lines come from the cache (restored from Drive), so
  only the new work runs. Repeat until every split is fully covered.
- **Fully-covered pairs are skipped** — you'll see
  `skip <model> / accumulate (no un-scored lines left in <dataset>)` and no
  commit for them.
- **Force a re-run** of one model: delete its `results/<run_id>/` dirs (and
  its `results/_cache/` entries) so those lines look un-scored again.
- **Add models / datasets:** edit `MODELS` / `DATASETS` and Run all — only
  the pairs with un-scored lines execute.
- Hosted models (Gemini/GPT/Claude): `02_colab_hosted_apis.ipynb`.